# 15개 실제 분류기만 학습하는 별도 구간검증


In [ ]:
from pathlib import Path
import os,ast,json,nbformat
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd();tables=root/'reports/tables'
setup=''.join(c.source for c in nbformat.read(root/'notebooks/145_rg3_shortshot_unlabeled_application.ipynb',4).cells if c.cell_type=='code')
base={'__name__':'__main__'};exec(setup[:setup.index('u_all=pd.read_csv')],base)
tree=ast.parse(setup[:setup.index('u_all=pd.read_csv')])
defs=[node.value.args[0].value for node in tree.body if isinstance(node,ast.Expr) and isinstance(node.value,ast.Call) and isinstance(node.value.func,ast.Name) and node.value.func.id=='exec' and isinstance(node.value.args[0],ast.Constant)]
def prediction_source(s):
 body=s[s.index('def predict('):]
 if 'def threshold(' in body:body=body[:body.index('def threshold(')]
 return body
A=prediction_source(defs[0]);B=prediction_source(defs[1]);D=prediction_source(defs[2]);C=prediction_source(base['sourceC'])
heads={}
def add(name,template,definition):
 ns=dict(template);ns['memo']={};ns['audit']=[]
 exec(definition,ns);heads[name]=ns
add('A9',base['na'],A)
add('A5',base['na'],A.replace('for C in [.1,1.]:','for C in [1.]:'))
add('B7',base['nb'],B)
add('B4',base['nb'],B.replace('for seed in [42,7]:','for seed in [42]:').replace('for C in [.1,1.]:','for C in [1.]:').replace("['rf','et','svm0.1','svm1.0','lr']","['rf','et','svm1.0','lr']"))
for seeds in [[42,7],[11,23],[101,37]]:
 add('C'+str(seeds),base['nc'][0],C.replace('for seed in [42,7]:','for seed in '+str(seeds)+':'))
add('C3',base['nc'][0],C.replace('for seed in [42,7]:','for seed in [42]:').replace('for C in [.1,1.]:','for C in []:').replace("['rf','et','svm0.1','svm1.0','lr']","['rf','et','lr']"))
add('D1',base['nd'],D)
b2=B.replace("for name,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:","for name,cls in []:").replace('for C in [.1,1.]:','for C in [1.]:').replace("['rf','et','svm0.1','svm1.0','lr']","['svm1.0','lr']")
c3=C.replace('for seed in [42,7]:','for seed in [42]:').replace('for C in [.1,1.]:','for C in [1.]:').replace("m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)","m=None").replace("out[view+'_lr']=m.predict_proba(b)[:,1]","out[view+'_lr']=np.zeros(len(q))").replace("['rf','et','svm0.1','svm1.0','lr']","['rf','et','svm1.0']")
add('B2',base['nb'],b2);add('C3svc',base['nc'][0],c3)
variants={'original38':('A9','B7','C21',38),'pruned15':('A9','B2','C3svc',15)}

lab=base['lab'];y=base['yn'];groups=base['gn'];use=base['usable'];dense=base['nc'][0]['dense']
barrel=dense.Barrel_Temperature_5__past10_std.to_numpy();hopper=-dense.Hopper_Temperature__dev10.to_numpy()
def head_score(name,ref,q):
 if name=='C21':return np.mean([head_score('C'+str(s),ref,q) for s in [[42,7],[11,23],[101,37]]],axis=0)
 if name=='C7':return head_score('C'+str([42,7]),ref,q)
 out=heads[name]['predict'](ref,q)
 if name.startswith('A'):return out['std24_geometric']
 if name.startswith('B'):return out['localphysical12_mean']
 if name.startswith('C'):return out['shock6_mean']
 return out
records=[];cutrows=[];audits=[];first=None
for block in pd.unique(groups):
 q=np.flatnonzero(groups==block);ref=np.flatnonzero((groups!=block)&use)
 normal=y[ref]==0
 bcut=float(np.quantile(barrel[ref][normal&np.isfinite(barrel[ref])],.95,method='higher'))
 hcut=float(np.quantile(hopper[ref][normal&np.isfinite(hopper[ref])],.95,method='higher'))
 outer={name:head_score(name,ref,q) for name in ['A9','B7','B2','C21','C3svc','D1']}
 inner={name:np.full(len(ref),np.nan) for name in outer}
 for ib in pd.unique(groups[ref]):
  mask=groups[ref]==ib
  for name in inner:inner[name][mask]=head_score(name,ref[~mask],ref[mask])
 assert all(np.isfinite(v).all() for v in inner.values())
 thresholds={}
 for name in inner:
  if name.startswith('A'):thresholds[name]=float(base['na']['threshold'](y[ref],inner[name],.03))
  elif name.startswith('B'):thresholds[name]=float(base['nb']['threshold'](y[ref],inner[name],.03))
  else:thresholds[name]=float(np.quantile(inner[name][normal],.995 if name=='D1' else .99,method='higher'))
 for label,(an,bn,cn,count) in variants.items():
  flags=np.column_stack([outer[an]>=thresholds[an],outer[bn]>=thresholds[bn],(outer[cn]>=thresholds[cn])&(barrel[q]>=bcut),(outer['D1']>=thresholds['D1'])&(hopper[q]>=hcut)])
  pred=flags.any(axis=1)
  cutrows.append(dict(candidate=label,block=block,A=thresholds[an],B=thresholds[bn],C=thresholds[cn],D=thresholds['D1'],barrel=bcut,hopper=hcut))
  for j,idx in enumerate(q):records.append(dict(candidate=label,classifiers=count,TimeStamp=lab.TimeStamp.iloc[idx],run_id=block,type=lab.type.iloc[idx],actual=int(y[idx]),prediction=int(pred[j]),A=int(flags[j,0]),B=int(flags[j,1]),C=int(flags[j,2]),D=int(flags[j,3])))
 audits.append(dict(block=block,train=len(ref),train_shortshot=int(y[ref].sum()),query=len(q),timestamp_overlap=len(set(lab.TimeStamp.iloc[ref])&set(lab.TimeStamp.iloc[q]))))
 print('Finished gradual shortshot reduction',block,flush=True)
 pd.DataFrame(records).to_csv(tables/'rg3_204_shortshot_reduction_progress.csv',index=False)
pp=pd.DataFrame(records);metrics=[]
for label,g in pp.groupby('candidate',sort=False):
 old=g[g.type.isin(['정상','미성형'])]
 tn,fp,fn,tp=confusion_matrix(old.actual,old.prediction,labels=[0,1]).ravel()
 metrics.append(dict(candidate=label,classifiers=int(g.classifiers.iloc[0]),N=len(old),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn))))
mm=pd.DataFrame(metrics)
saved=pd.read_csv(tables/'rg3_shortshot_143_fixed_candidate_predictions.csv',parse_dates=['TimeStamp'])
check=pp[pp.candidate.eq('original38')].merge(saved[['TimeStamp','prediction']],on='TimeStamp',suffixes=('_fresh','_original'),validate='one_to_one')
assert len(check)==591 and check.prediction_fresh.eq(check.prediction_original).all()
assert mm[mm.candidate.eq('original38')][['TP','FP','FN','TN']].iloc[0].tolist()==[6,11,2,555]
assert all(a['timestamp_overlap']==0 for a in audits)
pp.to_csv(tables/'rg3_204_shortshot_reduction_predictions.csv',index=False,encoding='utf-8-sig')
mm.to_csv(tables/'rg3_204_shortshot_reduction_metrics.csv',index=False)
pd.DataFrame(cutrows).to_csv(tables/'rg3_204_shortshot_reduction_thresholds.csv',index=False)
pd.DataFrame(audits).to_csv(tables/'rg3_204_shortshot_reduction_audit.csv',index=False)
print(mm.to_string(index=False))
text='# RG3 미성형0.48 후보 점진 축소\n\n원본38개A/B/C/D를실제로재학습해143의591샷예측모두일치확인. 각축소조합은학습내부구간검증에서A/B F1 및FP3%예산경계,C/D 정상분위경계와센서게이트를다시계산했다.\n\n'+mm.to_string(index=False)
text+='\n\n평가코호트정상566/미성형8=574샷. 가스17은학습/미성형단독평가에서제외했지만591전체쿼리예측을저장했다. 새독립시험아닌동일개발구간검증/축소후보비교. 기존저장모델미교체,commit/push없음. C7은열RF2+ET2+SVC2+LR1단일seed묶음;C3은RF1+ET1+LR1. A5는전체SVC1+2모드각SVC1/LR1;B4는RF1/ET1/SVC1/LR1.\n'
(root/'reports/rg3_204_shortshot_reduction.md').write_text(text,encoding='utf-8')


previous=pd.read_csv(tables/'rg3_shortshot_143_fixed_candidate_predictions.csv',parse_dates=['TimeStamp'])
now=pp[pp.candidate.eq('pruned15')]
merged=now.merge(previous[['TimeStamp','prediction']],on='TimeStamp',suffixes=('_direct','_shared'),validate='one_to_one')
assert len(merged)==591 and merged.prediction_direct.eq(merged.prediction_shared).all()
print('15 direct actual-pruned learners replay591 match:',True)

